# Experiment 1: End-to-End Fine-Tuning Pipeline Baseline (DistilGPT-2)

**Curriculum Focus**: Laboratory Experiment 1 Baseline Pipeline
**Model**: `distilbert/distilgpt2` (82M parameters)

### Objectives:
1. Implement a complete, working fine-tuning pipeline using Hugging Face `transformers`, `datasets`, and `Trainer`.
2. Perform train/test dataset splits, tokenization with fixed `max_length=128`, and padding/truncation.
3. Execute training with baseline `TrainingArguments` (2 epochs, batch size 2, lr 2e-5).
4. Compare before-and-after inference behavior.
5. **Pedagogical Takeaway (Viva)**: Demonstrate why full fine-tuning of generic causal language models is insufficient for structured instruction-following and JSON formatting, motivating the transition to instruction-tuned architectures (`Qwen/Qwen2.5-0.5B-Instruct`) with LoRA.

In [7]:
# Step 1: Environment Setup (Run in Google Colab with GPU/CPU)
!pip install -q transformers datasets accelerate torch tabulate


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: C:\Users\mahir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Execution Device: {device}')

Execution Device: cuda


## 1. Load Dataset
We load training pairs from `data/finetune/train.jsonl`.

In [9]:
data_path = 'data/finetune/train.jsonl'
# If running in standalone Colab, create sample fallback data if file is not cloned
if not os.path.exists(data_path):
    sample_data = [
        {'text': 'User: What is LoRA? Assistant: LoRA decomposes weight updates into low-rank matrices B and A.'},
        {'text': 'User: What does QLoRA do? Assistant: QLoRA quantizes base weights into 4-bit NF4 format.'},
        {'text': 'User: What is FAISS? Assistant: FAISS is a library for dense vector similarity search.'},
        {'text': 'User: What is LangGraph? Assistant: LangGraph manages stateful cyclic workflows with LLMs.'}
    ] * 25
else:
    with open(data_path, 'r', encoding='utf-8') as f:
        raw = [json.loads(line) for line in f]
    sample_data = []
    for item in raw:
        msgs = item.get('messages', [])
        u = next((m['content'] for m in msgs if m['role'] == 'user'), '')
        a = next((m['content'] for m in msgs if m['role'] == 'assistant'), '')
        sample_data.append({'text': f'User: {u}\nAssistant: {a}'})

dataset = Dataset.from_list(sample_data)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f'Train samples: {len(split_dataset["train"])}, Test samples: {len(split_dataset["test"])}')

Train samples: 90, Test samples: 10


## 2. Tokenization and Data Preprocessing
We tokenize with fixed `max_length=128` and set `pad_token = eos_token`.

In [10]:
model_name = 'distilbert/distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )

tokenized_train = split_dataset['train'].map(tokenize_function, batched=True)
tokenized_test = split_dataset['test'].map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

## 3. Baseline Model Loading (Before Fine-Tuning Inference)
Test base DistilGPT-2 before any training.

In [11]:
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

test_prompt = 'User: What is LoRA?\nAssistant:'
inputs = tokenizer(test_prompt, return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id)
print('--- BEFORE FINE-TUNING COMPLETION ---')
print(tokenizer.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

--- BEFORE FINE-TUNING COMPLETION ---
User: What is LoRA?
Assistant: I'm a professional, and I'm a professional. I'm a professional. I'm a professional. I'm a professional. I'm a professional. I'm a professional. I'm a professional


## 4. TrainingArguments and Trainer Setup
We configure standard baseline training arguments (2 epochs, batch size 2, lr 2e-5).

In [12]:
training_args = TrainingArguments(
    output_dir='./results_exp1',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    fp16=(device == 'cuda'),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

train_res = trainer.train()
print('Training Loss:', train_res.training_loss)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.493782,1.553733
2,0.752822,0.510115


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss: 2.322525045606825


## 5. After Fine-Tuning Inference & Evaluation

In [13]:
model.eval()
with torch.no_grad():
    out_after = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id)
print('--- AFTER FINE-TUNING COMPLETION ---')
print(tokenizer.decode(out_after[0], skip_special_tokens=True))

--- AFTER FINE-TUNING COMPLETION ---
User: What is LoRA?
Assistant: LoRA decomposes weight updates into low-rank matrices B and A.
: What is LoRA? Assistant: LoRA decomposes weight updates into low-rank matrices B and A


## 6. Viva Discussion & Limitations of Generic Base Fine-Tuning

| Aspect | DistilGPT-2 Baseline (Exp 1) | Qwen2.5-0.5B-Instruct + LoRA (Exp 2/3) |
|---|---|---|
| **Parameter Tuning** | Full model tuning (all 82M weights) | Parameter-efficient (PEFT LoRA, <0.5% weights) |
| **Instruction Capability** | Poor (completion only, repeats phrases) | Native chat following via ChatML templates |
| **JSON Schema Output** | Inconsistent, prone to syntax truncation | Strict structured output suitable for quiz generation |
| **Catastrophic Forgetting**| High risk due to full weight updates | Zero forgetting of base reasoning (weights frozen) |